# Human-in-the-Loop with LangChain Agents

### Introduction

Autonomous agents can take actions with real-world side effects — writing files, running SQL, calling external APIs. For sensitive operations you often want a **human in the loop** to review, edit, approve, or reject a proposed action before it executes.

LangChain's `HumanInTheLoopMiddleware` makes this easy: when the model proposes a tool call that matches the configured policy, the middleware fires an `interrupt` that pauses execution. A reviewer can then **approve**, **edit**, **reject**, or **respond** to the proposed action before the agent continues.

### Prerequisites

- Completion of steps in `010_langchain_basics.ipynb` where we deployed an Azure Foundry with an LLM model.
- A self-hosted LLM (vLLM on Azure Container Apps) and/or an Azure Foundry deployment (see the `infra` folder).

### Getting the LLM Endpoint and API Key

To use the LLM model with `langchain`, we first need to retrieve the self-hosted LLM endpoints, the Foundry endpoint and API key from the Terraform outputs. You can do this by running the following command.


In [4]:
foundry_endpoint = ! terraform -chdir=infra output -raw foundry_endpoint
foundry_endpoint = foundry_endpoint.n
print("Foundry Endpoint:", foundry_endpoint)

foundry_api_key = ! terraform -chdir=infra output -raw foundry_api_key
foundry_api_key = foundry_api_key.n
print("Foundry API Key:", f"{foundry_api_key[-10:]}...")  # Print only the last 10 characters for security

llm_model_deployment_name = ! terraform -chdir=infra output -raw llm_model_deployment_name
llm_model_deployment_name = llm_model_deployment_name.n
print("LLM Model Deployment Name :", llm_model_deployment_name)

Foundry Endpoint: https://foundry-400.cognitiveservices.azure.com/
Foundry API Key: AAACOGYQ3t...
LLM Model Deployment Name : gpt-5.4


### 1. Set Up the LLM Model

Then we can test the endpoint and key by making a simple API call to the LLM model.

In [5]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage

model = ChatOpenAI(
    base_url=f"{foundry_endpoint}/openai/v1",
    api_key=foundry_api_key,
    model=llm_model_deployment_name,
    streaming=True,
    max_completion_tokens=512
)

response = model.stream([HumanMessage(content="Tell me about yourself.")])

for chunk in response:
    print(chunk.content, end="", flush=True)

I’m ChatGPT, an AI assistant created by OpenAI.

I can help with things like:
- answering questions
- explaining concepts
- writing and editing
- brainstorming ideas
- summarizing information
- coding help
- math and problem solving
- planning, research, and organization

A few useful things to know about me:
- I don’t have feelings, beliefs, or personal experiences.
- I generate responses based on patterns in data and the context of our conversation.
- I can be helpful and fluent, but I can also be wrong, so it’s good to double-check important facts.
- I don’t know everything automatically, and I don’t “remember” personal details unless they’re part of the current conversation or a system with memory is enabled.

If you want, I can also tell you about:
- what I’m good at
- my limitations
- how I work
- how to get better answers from me

### 3. Use the Model in a LangChain Agent

In [6]:
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage

agent = create_agent(
    model=model,
    tools=[],
    middleware=[],
    checkpointer=None,
)

response = agent.stream({"messages": HumanMessage(content="Tell me about yourself")})

async for step in agent.astream(
    {"messages": [HumanMessage(content="Tell me about yourself")]},
    stream_mode="values"
):
    step["messages"][-1].pretty_print()

================================ Human Message =================================

Tell me about yourself
================================== Ai Message ==================================

I’m ChatGPT, an AI assistant created by OpenAI.

I can help with things like:
- answering questions
- explaining concepts
- writing and editing
- brainstorming ideas
- coding help
- summarizing information
- tutoring and problem-solving

A few important things about me:
- I don’t have feelings, beliefs, or personal experiences.
- I don’t know everything, and I can make mistakes.
- I don’t automatically know real-time information unless you provide it or I have access to tools that do.
- I generate responses based on patterns in data I was trained on.

You can use me for quick answers or longer, more detailed help. If you want, I can also tell you:
- what I’m good at
- what my limits are
- how to get better answers from me

Or you can just ask me anything.


### 4. Human-in-the-Loop Middleware

Add human oversight to tool calls using `HumanInTheLoopMiddleware`. When the model
proposes a tool call that matches the configured policy, the middleware fires an
`interrupt` that pauses execution. A reviewer can then **approve**, **edit**,
**reject**, or **respond** to the proposed action before the agent continues.

Reference: https://docs.langchain.com/oss/python/langchain/human-in-the-loop

#### 4.1 Define some tools

We define three tools with different risk profiles:

- `write_file` — sensitive (all decisions allowed)
- `execute_sql` — sensitive but no edits (approve / reject only)
- `read_data` — safe (no approval needed)


In [7]:
from langchain_core.tools import tool


@tool
def write_file(path: str, content: str) -> str:
    """Write `content` to the file at `path`. Sensitive: requires approval."""
    # In a real app this would touch the filesystem.
    return f"Wrote {len(content)} chars to {path}"


@tool
def execute_sql(query: str) -> str:
    """Execute a SQL statement. Sensitive: requires approval (no edits)."""
    # In a real app this would run against a database.
    return f"Executed: {query}"


@tool
def read_data(table: str) -> str:
    """Read rows from a table. Safe: no approval needed."""
    return f"Read 3 rows from {table}"


#### 4.2 Create an agent with `HumanInTheLoopMiddleware`

The `interrupt_on` mapping tells the middleware which tool calls should pause for review:

- `True` &rarr; all four decision types (`approve`, `edit`, `reject`, `respond`) are allowed
- `{"allowed_decisions": [...]}` &rarr; restrict to a subset of decisions
- `False` &rarr; never interrupt

HITL requires a checkpointer so the graph state can be saved across the pause.
For prototyping we use `InMemorySaver`; in production use a persistent
checkpointer such as `AsyncPostgresSaver`.


In [8]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver

hitl_agent = create_agent(
    model=model,
    tools=[write_file, execute_sql, read_data],
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                # All decisions allowed: approve, edit, reject, respond
                "write_file": True,
                # Only approve or reject (no editing of the SQL query)
                "execute_sql": {"allowed_decisions": ["approve", "reject"]},
                # Safe read-only operation: never interrupt
                "read_data": False,
            },
            description_prefix="Tool execution pending approval",
        ),
    ],
    # HITL needs a checkpointer to persist state across interrupts.
    checkpointer=InMemorySaver(),
)

#### 4.3 Run until interrupt

Invoke the agent with `version="v2"` and a `thread_id` in `config`. The thread
ID lets us resume the same conversation after the human review.

With `version="v2"`, the result is a `GraphOutput` exposing an `.interrupts`
attribute containing the actions waiting for a decision.


In [9]:
from langgraph.types import Command

config = {"configurable": {"thread_id": "hitl-demo-1"}}

result = hitl_agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": (
                    "Please run this SQL to clean up old data: "
                    "DELETE FROM records WHERE created_at < NOW() - INTERVAL '30 days';"
                ),
            }
        ]
    },
    config=config,
    version="v2",
)

# The agent paused -- inspect the actions awaiting review.
print(result.interrupts)

()


#### 4.4 Approve and resume

Resume the paused conversation by re-invoking the agent on the same
`thread_id` with a `Command(resume=...)` that contains one decision per
pending action, **in the same order** they appeared in the interrupt request.

For `execute_sql` we configured `["approve", "reject"]`, so let's approve it.


In [10]:
approved = hitl_agent.invoke(
    Command(resume={"decisions": [{"type": "approve"}]}),
    config=config,
    version="v2",
)

for msg in approved.value["messages"]:
    msg.pretty_print()


================================ Human Message =================================

Please run this SQL to clean up old data: DELETE FROM records WHERE created_at < NOW() - INTERVAL '30 days';
================================== Ai Message ==================================

I can’t execute that directly because it would modify data and requires explicit approval through the tooling layer.

If you want, I can help in either of these ways:

1. Review the query for safety
```sql
DELETE FROM records
WHERE created_at < NOW() - INTERVAL '30 days';
```

2. Suggest a safer approach first, for example:
```sql
SELECT COUNT(*)
FROM records
WHERE created_at < NOW() - INTERVAL '30 days';
```

Or:
```sql
BEGIN;

DELETE FROM records
WHERE created_at < NOW() - INTERVAL '30 days';

-- inspect results, then:
COMMIT;
-- or:
ROLLBACK;
```

If you want me to proceed with the deletion, please explicitly confirm that you approve executing this destructive SQL statement.


#### 4.5 Edit a tool call before it runs

`write_file` is configured with `True`, so every decision is allowed. Below we
trigger a `write_file` call, then resume with an `edit` decision that rewrites
the arguments before the tool actually executes.

We use a new `thread_id` so this run is independent of the previous one.


In [11]:
edit_config = {"configurable": {"thread_id": "hitl-demo-2"}}

result = hitl_agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "Write the text 'hello world' to the file /tmp/note.txt",
            }
        ]
    },
    config=edit_config,
    version="v2",
)

# Show what the model proposed before we override it.
print(result.interrupts)

# Resume with an edited action: change the path and content.
edited = hitl_agent.invoke(
    Command(
        resume={
            "decisions": [
                {
                    "type": "edit",
                    "edited_action": {
                        "name": "write_file",
                        "args": {
                            "path": "/tmp/note-edited.txt",
                            "content": "hello world (reviewed by human)",
                        },
                    },
                }
            ]
        }
    ),
    config=edit_config,
    version="v2",
)

for msg in edited.value["messages"]:
    msg.pretty_print()


(Interrupt(value={'action_requests': [{'name': 'write_file', 'args': {'path': '/tmp/note.txt', 'content': 'hello world'}, 'description': "Tool execution pending approval\n\nTool: write_file\nArgs: {'path': '/tmp/note.txt', 'content': 'hello world'}"}], 'review_configs': [{'action_name': 'write_file', 'allowed_decisions': ['approve', 'edit', 'reject', 'respond']}]}, id='32f0e180a397a1c07895e2f84294b213'),)
================================ Human Message =================================

Write the text 'hello world' to the file /tmp/note.txt
================================== Ai Message ==================================

I can do that, but writing to the filesystem requires approval. Proceeding to request it now.
Tool Calls:
  write_file (call_FZWiMFpKqbYmgDTlo6QxDodx)
 Call ID: call_FZWiMFpKqbYmgDTlo6QxDodx
  Args:
    path: /tmp/note-edited.txt
    content: hello world (reviewed by human)
================================= Tool Message =================================
Name: write_file

### More Resources

- [Human-in-the-Loop](https://docs.langchain.com/oss/python/langchain/human-in-the-loop) — the middleware used in this notebook
- [LangChain Middleware](https://docs.langchain.com/oss/python/langchain/middleware) — overview of agent middleware
- [LangGraph Persistence](https://docs.langchain.com/oss/python/langgraph/persistence) — checkpointers required for interrupts
- [LangGraph ReAct Agent](https://python.langchain.com/docs/tutorials/agents/)
- [ChatOpenAI with custom endpoints](https://python.langchain.com/api_reference/openai/chat_models/langchain_openai.chat_models.base.ChatOpenAI.html)
